In [84]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob 
import os
import pyarrow

The first step of the data cleaning is to check for basic issues, using a test sample of just one of the datasets.

In [86]:
df = pd.read_csv('data/2024-01/2024-01-metropolitan-street.csv')
df.head()
df.shape #91411 rows, 12 columns
df.info() #Shows that there are null and non-unique values for the supposed primary key
df['Crime ID'].nunique() #each row represents an individual crime (linked to 1 or more individuals) at each stage of the update process. Crime IDs reappear when outcome is updated. Many missing crime IDs, too.
df['Reported by'].nunique() #only 1 unique value
df['Falls within'].nunique() #only 1 unique value
#other conclusions: 'Context' is completely empty and can be discarded, can also discard 'Falls Within', 'Location', and ' LSOA code' for this analysis

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91411 entries, 0 to 91410
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               74897 non-null  object 
 1   Month                  91411 non-null  object 
 2   Reported by            91411 non-null  object 
 3   Falls within           91411 non-null  object 
 4   Longitude              90739 non-null  float64
 5   Latitude               90739 non-null  float64
 6   Location               91411 non-null  object 
 7   LSOA code              90739 non-null  object 
 8   LSOA name              90739 non-null  object 
 9   Crime type             91411 non-null  object 
 10  Last outcome category  74897 non-null  object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 8.4+ MB


1

The next step is to determine the missing data type of Crime ID.

In [88]:
df_test = pd.read_csv('data/2024-01/2024-01-metropolitan-street.csv')
df_test.loc[df_test['Crime ID'].isnull(), 'Crime type'].value_counts(dropna=False) 
df_test.loc[df_test['Crime ID'].isnull(), 'Last outcome category'].value_counts(dropna=False) 
df_test.loc[df_test['Crime type'] == 'Anti-social behaviour', 'Crime ID'].value_counts(dropna=False) 

#shows that all crimes with missing crime ID are just instances of antisocial behvaviour, an example of MAR. Also, all instances of antisocial behaviour have no crime ID.

Crime ID
NaN    16514
Name: count, dtype: int64

The next step is to ensure each crime ID is unique so each row represents one unique crime with its final update. 

In [90]:
df_test['Month'] = pd.to_datetime(df_test['Month']) #change month column datatype to date rather than string
df_test = df_test.sort_values('Month', ascending=True)

#separate the dataset into crimes with IDs and without.

with_id = df_test[df_test['Crime ID'].notna()]
no_id = df_test[df_test['Crime ID'].isna()]

#remove all duplicated crime IDs to show only the final updated outcome

with_id_clean = with_id.drop_duplicates(subset='Crime ID', keep='last')

#As shown above, non crime IDs (ASB) outcomes do not get updated, so are already unique instances of a crime. This does mean though that can't be 100% sure if non-crimeID duplicate rows are duplicate errors, or legitimate.
#re-merge the datasets.
df_test = pd.concat([with_id_clean, no_id], ignore_index=True)

print(f"Test merge complete. Total rows: {len(df_test)}")
#Final dataset should contain 88385 rows: the 16514 non-crimeID rows, and the 71781 rows with unique crime IDs (with most recent outcome listed)

Test merge complete. Total rows: 88385


The next step is to practise concatenating all the datasets together. To test intially, 4 datasets are used.

In [92]:
test_path = 'data/2023-12/'
test_files = glob.glob(os.path.join(test_path, "*.csv")) #grabs all the CSVs in the folder
keep_cols = ['Crime ID', 'Month', 'Reported by', 'Longitude', 'Latitude', 'LSOA name', 'Crime type', 'Last outcome category'] #relevant columns I have chosen
df_list = [pd.read_csv(f, usecols=keep_cols) for f in test_files] #creates a list of the datasets to be concatenated
df_test = pd.concat(df_list, ignore_index=True) 

print(f"Test merge complete. Total rows: {len(df_test)}") #should be 147,968

df_test['Month'] = pd.to_datetime(df_test['Month']) 
df_test = df_test.sort_values('Month', ascending=True)

with_id = df_test[df_test['Crime ID'].notnull()]
no_id = df_test[df_test['Crime ID'].isnull()]

with_id_clean = with_id.drop_duplicates(subset='Crime ID', keep='last')

df_test = pd.concat([with_id_clean, no_id], ignore_index=True)

print(f"Test merge complete. Total rows: {len(df_test)}") #should be 124536(unique crime IDs in with_id)+20319(ASB entries)=144855

#the code above works. Now time to look at this merged dataset overall

Test merge complete. Total rows: 147968
Test merge complete. Total rows: 144855


Now, all 96 datasets can be concatenated and cleaned, and missing values determined.

In [94]:
files_path = 'data/'
files = glob.glob(os.path.join(files_path, "**", "*.csv"), recursive=True)
keep_cols = ['Crime ID', 'Month', 'Reported by', 'Longitude', 'Latitude', 'LSOA name', 'Crime type', 'Last outcome category'] 
df_merged = pd.concat((pd.read_csv(f, usecols=keep_cols) for f in files), ignore_index=True) #quicker than using list comprehension for this many files

In [96]:
df_merged.info()
df_merged['Month'] = pd.to_datetime(df_merged['Month']) 
df_merged = df_merged.sort_values('Month', ascending=True)

with_id = df_merged[df_merged['Crime ID'].notnull()]
no_id = df_merged[df_merged['Crime ID'].isnull()]

print(with_id['Crime ID'].nunique())
print(len(no_id))
print(len(with_id))

with_id_clean = with_id.drop_duplicates(subset='Crime ID', keep='last') #13,050 duplicates removed from with_id

df_merged = pd.concat([with_id_clean, no_id], ignore_index=True)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3665530 entries, 0 to 3665529
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Crime ID               object 
 1   Month                  object 
 2   Reported by            object 
 3   Longitude              float64
 4   Latitude               float64
 5   LSOA name              object 
 6   Crime type             object 
 7   Last outcome category  object 
dtypes: float64(2), object(6)
memory usage: 223.7+ MB
3042350
610130
3055400
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652480 entries, 0 to 3652479
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Crime ID               object        
 1   Month                  datetime64[ns]
 2   Reported by            object        
 3   Longitude              float64       
 4   Latitude               float64       
 5   LSOA name              object    

Now, the missing values in other columns than the primary key are determined.

In [98]:
df_merged.isnull().sum() #21861 longitude and latitude values are missing, and 21865 LSOA names. Everything else intact. 

#could fill in missing values here for crime IDs with 'ASB' and last outcome category for ASB 

df_merged.loc[df_merged['Longitude'].isnull(), ['Latitude', 'LSOA name']].value_counts(dropna=False) #this shows that every row missing longitude is also missing latitude and LSOA name
df_merged.loc[df_merged['LSOA name'].isnull(), ['Crime ID', 'Latitude', 'Longitude', 'Month', 'Reported by']].value_counts() #4 rows that are missing LSOA but not geospatial variables

Crime ID                                                          Latitude   Longitude  Month       Reported by                
313acb161af3eae87ab6569441726be06c9d1906e24c962d4fbf4552f7e054d8  54.585701  -6.005104  2024-11-01  Sussex Police                  1
4ac3a5e54fc38403a6dd0d9624a8b31c57ea7e1d812ff8e9f850cba5296ce37c  54.589707  -6.789849  2024-11-01  Metropolitan Police Service    1
a3f00b306b63807392f6e46b0b3cdff15e323f0573dea332c4d368ac4683d04d  54.428177  -6.581064  2024-05-01  Sussex Police                  1
b0877d237222c11fab4ca0669e51e41c9f035889bb1ef6bd45c6fa59e1c2e1f8  54.833107  -6.339279  2024-11-01  Sussex Police                  1
Name: count, dtype: int64

In [67]:
df_merged['Month'].value_counts(normalize=True) #total ocurrences of months throughout the dataset

Month
2025-07-01    0.046388
2024-07-01    0.045171
2025-06-01    0.044022
2024-10-01    0.044017
2024-05-01    0.043863
2025-05-01    0.043654
2024-08-01    0.043355
2024-06-01    0.043317
2025-08-01    0.043144
2025-10-01    0.042388
2024-11-01    0.042011
2025-03-01    0.041671
2024-09-01    0.041471
2025-04-01    0.041219
2024-03-01    0.040939
2024-04-01    0.040615
2025-09-01    0.040561
2025-11-01    0.040512
2023-12-01    0.039659
2024-01-01    0.039369
2024-12-01    0.039263
2024-02-01    0.038490
2025-01-01    0.038179
2025-02-01    0.036722
Name: proportion, dtype: float64

In [65]:
df_merged.loc[df_merged['Longitude'].isnull(), 'Month'].value_counts(normalize=True) #ocurrences of missing geospatial data for each month - if MCAR, these should reflect the overall proportions given above.
#Conclusion: Month is not a factor 

Month
2024-01-01    0.062486
2023-12-01    0.061800
2024-10-01    0.048671
2024-07-01    0.046704
2024-08-01    0.045927
2024-11-01    0.045195
2024-05-01    0.044646
2024-09-01    0.044326
2025-10-01    0.044005
2024-06-01    0.042953
2025-09-01    0.041855
2025-11-01    0.040849
2024-12-01    0.040071
2025-03-01    0.039797
2025-07-01    0.039339
2025-08-01    0.038608
2025-02-01    0.038287
2024-04-01    0.034857
2025-04-01    0.033896
2025-05-01    0.033896
2024-03-01    0.033393
2025-01-01    0.033210
2025-06-01    0.032661
2024-02-01    0.032569
Name: proportion, dtype: float64

In [69]:
df_merged['Reported by'].value_counts(normalize=True) 

Reported by
Metropolitan Police Service    0.621291
West Midlands Police           0.182919
Thames Valley Police           0.107765
Sussex Police                  0.088025
Name: proportion, dtype: float64

In [63]:
df_merged.loc[df_merged['Longitude'].isnull(), 'Reported by'].value_counts(normalize=True) 
#Conclusion: MAR, sussex and Thames Valley more likely to have missing geospatial data.
#West Midlands have no missing geospatial data at all
#If removing all these rows, more likely to be removing sussex and thames rows...will underestimate crimes happening there
#If keeping in, just remember to factor it into analysis

Reported by
Sussex Police                  0.554869
Thames Valley Police           0.383606
Metropolitan Police Service    0.061525
Name: proportion, dtype: float64

In [71]:
df_merged['Crime type'].value_counts(normalize=True) 

Crime type
Violence and sexual offences    0.284705
Anti-social behaviour           0.167045
Shoplifting                     0.084055
Other theft                     0.083975
Vehicle crime                   0.076394
Criminal damage and arson       0.058697
Public order                    0.055615
Theft from the person           0.053893
Burglary                        0.043048
Drugs                           0.034842
Robbery                         0.023366
Other crime                     0.014342
Bicycle theft                   0.011607
Possession of weapons           0.008416
Name: proportion, dtype: float64

In [73]:
df_merged.loc[df_merged['Longitude'].isnull(), 'Crime type'].value_counts(normalize=True)
#Conclusions: MNAR, because the location values are missing BECAUSE 



#need to explain this further.....


#Can do additional statistical and coding checks here

Crime type
Violence and sexual offences    0.428663
Public order                    0.077947
Other theft                     0.077901
Shoplifting                     0.074516
Drugs                           0.067609
Criminal damage and arson       0.060702
Other crime                     0.048534
Vehicle crime                   0.042862
Anti-social behaviour           0.033164
Theft from the person           0.028041
Burglary                        0.021271
Possession of weapons           0.019441
Robbery                         0.010704
Bicycle theft                   0.008646
Name: proportion, dtype: float64

In [75]:
df_merged['Last outcome category'].value_counts(normalize=True) 

Last outcome category
Investigation complete; no suspect identified          0.488594
Unable to prosecute suspect                            0.332460
Status update unavailable                              0.052798
Court result unavailable                               0.035568
Local resolution                                       0.025262
Awaiting court outcome                                 0.023115
Under investigation                                    0.019880
Action to be taken by another organisation             0.010735
Offender given a caution                               0.005232
Further investigation is not in the public interest    0.002409
Offender given penalty notice                          0.001643
Formal action is not in the public interest            0.001343
Further action is not in the public interest           0.000663
Suspect charged as part of another case                0.000295
Offender given a drugs possession warning              0.000002
Name: proportion, 

In [77]:
df_merged.loc[df_merged['Longitude'].isnull(), 'Last outcome category'].value_counts(normalize=True, dropna=False)

Last outcome category
Investigation complete; no suspect identified          0.334706
Unable to prosecute suspect                            0.265679
Status update unavailable                              0.188829
Local resolution                                       0.045881
Court result unavailable                               0.040300
Under investigation                                    0.034079
NaN                                                    0.033164
Awaiting court outcome                                 0.017611
Action to be taken by another organisation             0.016468
Offender given a caution                               0.008783
Further investigation is not in the public interest    0.007776
Formal action is not in the public interest            0.002424
Further action is not in the public interest           0.002196
Suspect charged as part of another case                0.001327
Offender given penalty notice                          0.000778
Name: proportion, 

Finally, save the file as a Parquet file, ready for EDA analysis